In [39]:
!pip install -q langgraph langchain langchain-google-genai langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 46.0 MB/s eta 0:00:00


In [55]:
from google.colab import userdata

OPENROUTER_API_KEY = userdata.get("OPEN_ROUTER_API_KEY")

print("API Loaded")

API Loaded


In [60]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="mistral-medium-3.5",
    openai_api_key=OPENROUTER_API_KEY,
    openai_api_base="https://openrouter.ai/api/v1",
    temperature=0.1
)

In [5]:
response = llm.invoke("Greet")

print(response.content)

Hello! How can I help you today?


In [6]:
from typing import TypedDict


In [7]:
class AgentState(TypedDict):
    problem: str
    analysis: str
    generated_code: str
    solved: bool

In [8]:
def analyze_problem(state):

    prompt = f"""
You are a competitive programming expert.

Analyze this problem briefly.

Problem:
{state["problem"]}

Give:
1. Key observation
2. Algorithm idea
"""

    response = llm.invoke(prompt)

    state["analysis"] = response.content

    return state

In [10]:
def generate_code(state):

    prompt = f"""
You are a competitive programming expert.

Problem:
{state["problem"]}

Analysis:
{state["analysis"]}

Generate ONLY C++ code.

No explanation.
"""

    response = llm.invoke(prompt)

    state["generated_code"] = (
    response.content
    .replace("```C++", "")
    .replace("```", "")
    .strip()
)

    return state

In [11]:

def evaluate_code(state):

    prompt = f"""
You are a competitive programming judge.

Problem:
{state["problem"]}

Code:
{state["generated_code"]}

Is this solution logically correct?

Answer only:

YES

or

NO
"""

    response = llm.invoke(prompt)

    verdict = response.content.strip()

    state["solved"] = "YES" in verdict.upper()

    return state

In [12]:
def improve_code(state):

    prompt = f"""
You are an expert competitive programmer.

Problem:
{state["problem"]}

Previous Code:
{state["generated_code"]}

The previous solution may contain mistakes.

Generate an improved C++ solution.

Return ONLY C++ code.
Do not explain.
Do not use markdown.
"""

    response = llm.invoke(prompt)

    state["generated_code"] = (
    response.content
    .replace("```C++", "")
    .replace("```", "")
    .strip()
)

    return state

In [13]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

In [14]:
workflow.add_node("analyze", analyze_problem)
workflow.add_node("generate", generate_code)
workflow.add_node("evaluate", evaluate_code)
workflow.add_node("improve", improve_code)


In [15]:
print("workflow" in globals())

True


In [16]:
def should_continue(state):

    if state["solved"]:
        return "end"

    return "improve"

In [17]:
workflow.set_entry_point("analyze")

workflow.add_edge("analyze", "generate")
workflow.add_edge("generate", "evaluate")

In [18]:
workflow.add_conditional_edges(
    "evaluate",
    should_continue,
    {
        "end": END,
        "improve": "improve"
    }
)

workflow.add_edge("improve", "evaluate")

In [19]:
graph = workflow.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


Problem 1:Haloumi Boxes

In [20]:
state = {
    "problem": """
A. Halloumi Boxes

Theofanis is busy after his last contest, as now, he has to deliver many halloumis all over the world. He stored them inside n boxes and each box has some number ai written on it.

He wants to sort them in non-decreasing order by their number, however, his machine works in a strange way. It can only reverse any subarray of boxes with length at most k.

Find if it's possible to sort the boxes using any number of reverses.

Input:
The first line contains t — the number of test cases.

The first line of each test case contains two integers n and k.

The second line contains n integers a1, a2, ..., an.

Output:
For each test case print YES if the array can be sorted, otherwise NO.
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [21]:
result = graph.invoke(state)

In [22]:
print(result["generated_code"])

cpp
#include <iostream>
#include <vector>
#include <algorithm> // Not strictly needed for this solution, but good practice for competitive programming

void solve() {
    int n;
    long long k; // k can be large, though only k=1 vs k>=2 matters
    std::cin >> n >> k;
    std::vector<int> a(n);
    for (int i = 0; i < n; ++i) {
        std::cin >> a[i];
    }

    if (k >= 2) {
        std::cout << "YES\n";
    } else { // k == 1
        bool sorted = true;
        for (int i = 0; i < n - 1; ++i) {
            if (a[i] > a[i+1]) {
                sorted = false;
                break;
            }
        }
        if (sorted) {
            std::cout << "YES\n";
        } else {
            std::cout << "NO\n";
        }
    }
}

int main() {
    std::ios_base::sync_with_stdio(false);
    std::cin.tie(NULL);
    int t;
    std::cin >> t;
    while (t--) {
        solve();
    }
    return 0;
}


In [23]:
print(result['solved'])

True


In [24]:
print(result)

{'problem': "\nA. Halloumi Boxes\n\nTheofanis is busy after his last contest, as now, he has to deliver many halloumis all over the world. He stored them inside n boxes and each box has some number ai written on it.\n\nHe wants to sort them in non-decreasing order by their number, however, his machine works in a strange way. It can only reverse any subarray of boxes with length at most k.\n\nFind if it's possible to sort the boxes using any number of reverses.\n\nInput:\nThe first line contains t — the number of test cases.\n\nThe first line of each test case contains two integers n and k.\n\nThe second line contains n integers a1, a2, ..., an.\n\nOutput:\nFor each test case print YES if the array can be sorted, otherwise NO.\n", 'analysis': 'Here\'s a brief analysis of the problem:\n\n1.  **Key Observation:**\n    The crucial insight lies in understanding the power of the allowed operation. If we can reverse *any* subarray of length 2, we can effectively swap any two adjacent elements

Problem 2:Tea Tasting

In [26]:
state={
    "problem": """
A.A tea manufacturer decided to conduct a massive tea tasting. n
 sorts of tea will be tasted by n
 tasters. Both the sorts of tea and the tasters are numbered from 1
 to n
. The manufacturer prepared ai
 milliliters of the i
-th sort of tea. The j
-th taster can drink bj
 milliliters of tea at once.

The tasting will be conducted in steps. During the first step, the i
-th taster tastes the i
-th sort of tea. The i
-th taster drinks min(ai,bi)
 tea (how much is available of the i
-th sort and how much the i
-th taster can drink). ai
 also decreases by this amount.

Then all tasters move to the previous sort of tea. Thus, during the second step, the i
-th taster tastes the (i−1)
-st sort of tea. The i
-th taster drinks min(ai−1,bi)
 tea. The 1
-st person ends the tasting.

During the third step, the i
-th taster tastes the (i−2)
-nd sort of tea. The 2
-nd taster ends the tasting. This goes on until everyone ends the tasting.

Take a look at the tasting process for n=3
, a=[10,20,15]
, b=[9,8,6]
. In the left row, there are the current amounts of each sort of tea. In the right column, there are current amounts of tea each taster has drunk in total. The arrow tells which taster each tea goes to on the current step. The number on the arrow is the amount — minimum of how much is available of the sort of tea and how much the taster can drink.

Input:
...
Output:
...
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [28]:
result=graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])


Solved: True
cpp
#include <iostream>
#include <vector>
#include <numeric>
#include <algorithm>

int main() {
    // Optimize C++ standard streams for competitive programming
    std::ios_base::sync_with_stdio(false);
    std::cin.tie(NULL);

    int n;
    std::cin >> n;

    std::vector<long long> a(n);
    for (int i = 0; i < n; ++i) {
        std::cin >> a[i];
    }

    std::vector<long long> b(n);
    for (int i = 0; i < n; ++i) {
        std::cin >> b[i];
    }

    // P_B[k] stores the sum of b[0]...b[k-1].
    // P_B has size n+1, with P_B[0] = 0.
    // P_B[k] represents the total drinking capacity of tasters 0 through k-1.
    std::vector<long long> P_B(n + 1, 0);
    for (int k = 0; k < n; ++k) {
        P_B[k + 1] = P_B[k] + b[k];
    }

    // diff_full_counts is a difference array.
    // diff_full_counts[j] stores the net change in the count of times taster j drinks their full b_j.
    // Size n+1 to handle index n for end_taster_idx + 1 when end_taster_idx is n-1.
    s

Problem 3:Lunar New Year and a Wander

In [63]:
state = {
    "problem": """
Lunar New Year is approaching, and Bob decides to take a wander in a nearby park.

The park can be represented as a connected graph with n
 nodes and m
 bidirectional edges. Initially Bob is at the node 1
 and he records 1
 on his notebook. He can wander from one node to another through those bidirectional edges. Whenever he visits a node not recorded on his notebook, he records it. After he visits all nodes at least once, he stops wandering, thus finally a permutation of nodes a1,a2,…,an
 is recorded.

Wandering is a boring thing, but solving problems is fascinating. Bob wants to know the lexicographically smallest sequence of nodes he can record while wandering. Bob thinks this problem is trivial, and he wants you to solve it.

A sequence x
 is lexicographically smaller than a sequence y
 if and only if one of the following holds:

x
 is a prefix of y
, but x≠y
 (this is impossible in this problem as all considered sequences have the same length);
in the first position where x
 and y
 differ, the sequence x
 has a smaller element than the corresponding element in y
.
Input
The first line contains two positive integers n
 and m
 (1≤n,m≤105
), denoting the number of nodes and edges, respectively.

The following m
 lines describe the bidirectional edges in the graph. The i
-th of these lines contains two integers ui
 and vi
 (1≤ui,vi≤n
), representing the nodes the i
-th edge connects.

Note that the graph can have multiple edges connecting the same two nodes and self-loops. It is guaranteed that the graph is connected.

Output
Output a line containing the lexicographically smallest sequence a1,a2,…,an
 Bob can record.
 """,
    "analysis": "",
    "generated_code": "",
    "solved": False
}

In [64]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])

BadRequestError: Error code: 400 - {'error': {'message': 'mistral-medium-3.5 is not a valid model ID', 'code': 400}, 'user_id': 'user_3DrQEPrVzaYVClEyTfXBbhYnxPN'}

Problem 3 Game with Integers

In [62]:
state = {
    "problem": """
A. Game with Integers

Vanya and Vova are playing a game. Players are given an integer n. On their turn, the player can add 1 to the current integer or subtract 1. The players take turns; Vanya starts. If after Vanya's move the integer is divisible by 3, then he wins. If 10 moves have passed and Vanya has not won, then Vova wins.

Write a program that determines who will win if both players play optimally.

Input

The first line contains the integer t (1 <= t <= 100) — the number of test cases.

The single line of each test case contains the integer n (1 <= n <= 1000).

Output

For each test case print "First" if Vanya wins and "Second" if Vova wins.

Example

Input
6
1
3
5
100
999
1000

Output
First
Second
First
First
Second
First
""",
    "analysis": "",
    "generated_code": "",
    "solved": False
}


In [61]:
result = graph.invoke(state)

print("Solved:", result["solved"])
print(result["generated_code"])


BadRequestError: Error code: 400 - {'error': {'message': 'mistral-medium-3.5 is not a valid model ID', 'code': 400}, 'user_id': 'user_3DrQEPrVzaYVClEyTfXBbhYnxPN'}